**Lab type:** debug

**Course:** ML201 — Applied Machine Learning

**Lesson:** Cross-Validation Strategies and Data Leakage

**Task:** The AI-generated analysis below contains 3 bugs. For each bug: identify what is wrong, explain why the output is misleading, and write the corrected code in the fix cell.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    cross_val_score, StratifiedKFold, TimeSeriesSplit, GridSearchCV
)

np.random.seed(42)

# Dataset 1: tabular credit risk classification
n = 1500
income = np.random.normal(55000, 20000, n).clip(15000, 150000)
credit_score = np.random.normal(660, 80, n).clip(300, 850)
debt_ratio = np.random.beta(2, 5, n)
employment_years = np.random.exponential(6, n).clip(0, 40)
age = np.random.normal(42, 12, n).clip(21, 75)
region_code = np.random.randint(0, 10, n)
account_type = np.random.randint(0, 3, n)
num_late_payments = np.random.poisson(1.2, n)

log_odds = (
    -2.8
    + 1.8 * debt_ratio
    - 0.004 * (credit_score - 660) / 80
    + 0.03 * num_late_payments
    - 0.00001 * income
)
prob_default = 1 / (1 + np.exp(-log_odds))
default_target = (np.random.rand(n) < prob_default).astype(int)

feature_cols = ['income', 'credit_score', 'debt_ratio', 'employment_years',
                'age', 'region_code', 'account_type', 'num_late_payments']
df = pd.DataFrame({
    'income': income, 'credit_score': credit_score, 'debt_ratio': debt_ratio,
    'employment_years': employment_years, 'age': age, 'region_code': region_code,
    'account_type': account_type, 'num_late_payments': num_late_payments,
    'default': default_target
})

# Dataset 2: time-series version (500 monthly records in temporal order)
n_ts = 500
dates = pd.date_range(start='2015-01-01', periods=n_ts, freq='MS')
ts_income = np.random.normal(55000, 20000, n_ts).clip(15000, 150000)
ts_credit = np.random.normal(660, 80, n_ts).clip(300, 850)
ts_debt = np.random.beta(2, 5, n_ts)
ts_emp = np.random.exponential(6, n_ts).clip(0, 40)
ts_age = np.random.normal(42, 12, n_ts).clip(21, 75)
ts_region = np.random.randint(0, 10, n_ts)
ts_acct = np.random.randint(0, 3, n_ts)
ts_late = np.random.poisson(1.2, n_ts)
ts_log_odds = -2.8 + 1.8 * ts_debt - 0.004 * (ts_credit - 660) / 80 + 0.03 * ts_late - 0.00001 * ts_income
ts_prob = 1 / (1 + np.exp(-ts_log_odds))
ts_target = (np.random.rand(n_ts) < ts_prob).astype(int)

df_ts = pd.DataFrame({
    'date': dates,
    'income': ts_income, 'credit_score': ts_credit, 'debt_ratio': ts_debt,
    'employment_years': ts_emp, 'age': ts_age, 'region_code': ts_region,
    'account_type': ts_acct, 'num_late_payments': ts_late,
    'default': ts_target
})
df_ts = df_ts.sort_values('date').reset_index(drop=True)

X = df[feature_cols].values
y = df['default'].values

print(f'df shape: {df.shape}')
print(f'df_ts shape: {df_ts.shape}')
print(f'df class balance:\n{df["default"].value_counts(normalize=True).round(3)}')
print(f'df_ts class balance:\n{df_ts["default"].value_counts(normalize=True).round(3)}')

## Step 1: Preprocessing before cross-validation

An analyst standardises the features and then evaluates with cross-validation.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)   # <- Bug 1: scaler fit on all data before CV splits

scores = cross_val_score(
    LogisticRegression(max_iter=1000),
    X_scaled, y,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='roc_auc'
)
print(f"CV AUC: {scores.mean():.3f} +/- {scores.std():.3f}")
print("Model evaluated correctly with cross-validation.")

**Bug 1 Investigation:** The scaler was fit on `X` before any CV split was created. Which statistics did `StandardScaler.fit_transform` compute, and were any of those statistics derived from the rows that later became each fold's validation set? Is the reported CV AUC an honest estimate of generalisation performance?

*(Write your answer here.)*

In [ ]:
# Fix Bug 1 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Step 2: Cross-validation for time-series data

An analyst runs 5-fold CV on the time-series dataset to estimate model performance for a system that predicts next month's churn.

In [ ]:
X_ts = df_ts[feature_cols].values
y_ts = df_ts['default'].values

scores_ts = cross_val_score(
    Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]),
    X_ts, y_ts,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),  # <- Bug 2: random fold assignment ignores time order
    scoring='roc_auc'
)
print(f"CV AUC: {scores_ts.mean():.3f} +/- {scores_ts.std():.3f}")
print("Model validated correctly for time-series prediction.")

**Bug 2 Investigation:** The records in `df_ts` are ordered chronologically — record 0 is the earliest month and record 499 is the latest. `StratifiedKFold` with `shuffle=True` assigns rows to folds randomly. If fold 3 is used as the validation set and its training data includes records from months 40–50 while the validation fold contains records from months 10–20, what problem does this create for a model that is supposed to predict *future* events?

*(Write your answer here.)*

In [ ]:
# Fix Bug 2 here
# YOUR CODE HERE
# Hint: use TimeSeriesSplit(n_splits=5) and print the training and validation index ranges
# for each fold to verify that training always precedes validation in time.

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Step 3: Reporting GridSearchCV best score as generalisation performance

An analyst runs `GridSearchCV` then reports the best CV score as the model's performance.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {'model__C': [0.01, 0.1, 1.0, 10.0]}
pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])

grid_search = GridSearchCV(pipe, param_grid, cv=5, scoring='roc_auc')
grid_search.fit(X, y)

print(f"Best C: {grid_search.best_params_}")
print(f"Best CV AUC: {grid_search.best_score_:.3f}")  # <- Bug 3: optimistically biased from hyperparameter selection
print("This is the model's true generalisation performance.")

**Bug 3 Investigation:** `grid_search.best_score_` is the cross-validated AUC of the *best* hyperparameter setting selected from 4 candidates. Even if all four values of `C` produced identical true performance, the best of 4 noisy CV scores would still be higher than any individual score just by chance. Why is this score a biased (optimistic) estimate of how the final model will perform on unseen data? How does the number of candidates in `param_grid` affect the magnitude of this bias?

*(Write your answer here.)*

In [ ]:
# Fix Bug 3 here
# YOUR CODE HERE
# Hint: nested CV — wrap the GridSearchCV object in cross_val_score with an outer
# StratifiedKFold(5, shuffle=True, random_state=42) to get an unbiased estimate.
# Use cv=3 as the inner fold for GridSearchCV to keep runtime reasonable.
# Print both grid_search.best_score_ and the nested CV score side by side.

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Corrected Analysis

The cell below applies all three fixes in the correct order. Run it end-to-end to confirm the pipeline is now sound.

In [ ]:
# =============================================================
# Fix 1: Pipeline prevents preprocessing leakage
# =============================================================
pipe_no_leak = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])
cv_correct = StratifiedKFold(5, shuffle=True, random_state=42)
scores_no_leak = cross_val_score(pipe_no_leak, X, y, cv=cv_correct, scoring='roc_auc')
print("[Fix 1] Pipeline CV AUC (no preprocessing leakage):")
print(f"  {scores_no_leak.mean():.3f} +/- {scores_no_leak.std():.3f}")

# =============================================================
# Fix 2: TimeSeriesSplit for temporal data
# =============================================================
tscv = TimeSeriesSplit(n_splits=5)
scores_ts_fixed = cross_val_score(
    Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]),
    X_ts, y_ts,
    cv=tscv,
    scoring='roc_auc'
)
print("\n[Fix 2] TimeSeriesSplit CV AUC (training always precedes validation):")
print(f"  {scores_ts_fixed.mean():.3f} +/- {scores_ts_fixed.std():.3f}")
print("  Fold index ranges:")
for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X_ts)):
    print(f"    Fold {fold_idx + 1}: train [{train_idx[0]}..{train_idx[-1]}]  val [{val_idx[0]}..{val_idx[-1]}]")

# =============================================================
# Fix 3: Nested CV for unbiased hyperparameter selection estimate
# =============================================================
inner_cv = StratifiedKFold(3, shuffle=True, random_state=42)
outer_cv = StratifiedKFold(5, shuffle=True, random_state=42)
param_grid = {'model__C': [0.01, 0.1, 1.0, 10.0]}
pipe_gs = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
grid_search_nested = GridSearchCV(pipe_gs, param_grid, cv=inner_cv, scoring='roc_auc')
nested_scores = cross_val_score(grid_search_nested, X, y, cv=outer_cv, scoring='roc_auc')

# Refit once to get best_score_ for comparison
grid_search_nested.fit(X, y)
print("\n[Fix 3] Nested CV vs GridSearchCV best score:")
print(f"  GridSearchCV best_score_ (biased) : {grid_search_nested.best_score_:.3f}")
print(f"  Nested CV AUC (unbiased)          : {nested_scores.mean():.3f} +/- {nested_scores.std():.3f}")
print(f"  Best C selected: {grid_search_nested.best_params_}")